In [0]:
%sql
use catalog sagar_cap3_cat1;
create schema if not exists ref;
create schema if not exists weather

In [0]:
# Set the Azure storage account credentials securely
storage_account_name = ''
storage_account_key=''
container=''
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)


In [0]:
file_names = ['cameras', 'corridors', 'holiday_calendar', 'intersections', 'sensors']
for file_name in file_names:
    print(f"Loading ref file: {file_name}.csv")
    ref_df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .load(f'abfss://{container}@{storage_account_name}.dfs.core.windows.net/synthetic_data/ref/{file_name}.csv')
    )
    ref_df.write.format('delta').mode('overwrite').saveAsTable(f'sagar_cap3_cat1.ref.{file_name}')

print("Reference data loaded successfully.")



Loading ref file: cameras.csv
Loading ref file: corridors.csv
Loading ref file: holiday_calendar.csv
Loading ref file: intersections.csv
Loading ref file: sensors.csv
Reference data loaded successfully.


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5202026059301463>, line 26
     17     weather_zone_path = f'abfss://{container}@{storage_account_name}.dfs.core.windows.net/synthetic_data/weather/date=2025-08-29/zone={zone}/weather_hourly.csv'
     19     weather_df = (
     20         spark.read
     21         .format('csv')
     22         .option('header', 'true')
     23         .load(weather_zone_path)
     24     )
---> 26     weather_df.write.format('delta').mode('append').saveAsTable('sagar_cap3_cat1.weather')
     28 print("Weather data loaded successfully.")

File /databricks/spark/python/pyspark/sql/connect/readwriter.py:713, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    711 self._write.table_name = name
    712 self._write.table_save_method = "save_as_table"
--> 713 _, _, ei = self._spark.client.execute_command(
    71

In [0]:
zones=['NORTHWEST', 'CENTRAL', 'NORTH', 'SOUTH', 'EAST', 'WEST']

for zone in zones:
    weather_zone_path = f'abfss://{container}@{storage_account_name}.dfs.core.windows.net/synthetic_data/weather/date=2025-08-29/zone={zone}/weather_hourly.csv'

    weather_df = (
        spark.read
        .format('csv')
        .option('header', 'true')
        .load(weather_zone_path)
    )

    weather_df.write.format('delta').mode('append').saveAsTable('sagar_cap3_cat1.weather.weather_all_zones')

print("Weather data loaded successfully.") 

Weather data loaded successfully.


In [0]:
%sql
use catalog sagar_cap3_cat1;
create schema if not exists stream;
use schema stream; 

In [0]:
import base64
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType, IntegerType


spark = SparkSession.builder.appName("EventHubStream").getOrCreate()


event_hub_connection_string = ""

eh_conf = {
    "eventhubs.connectionString": spark._jvm.org.apache.spark.eventhubs.EventHubsUtils.encrypt(event_hub_connection_string)
}

# Define schema
event_schema = StructType([
    StructField("sensor_id", StringType(), True),
    StructField("intersection_id", StringType(), True),
    StructField("zone", StringType(), True),
    StructField("corridor_id", StringType(), True),
    StructField("geo", StructType([
        StructField("lat", DoubleType(), True),
        StructField("lon", DoubleType(), True)
    ]), True),
    StructField("event_ts", TimestampType(), True),
    StructField("ingest_ts", TimestampType(), True),
    StructField("vehicle_count", IntegerType(), True),
    StructField("avg_speed_kmh", DoubleType(), True),
    StructField("lane_status", StringType(), True),
    StructField("aqi", IntegerType(), True)
])

# Read from Event Hub
df = (
    spark.readStream
        .format("eventhubs")
        .options(**eh_conf)
        .load()
)


df_parsed = (
    df.select(from_json(col("body").cast("string"), event_schema).alias("data"))
      .select("data.*")
)


(
    df_parsed.writeStream
      .format("delta")
      .option("checkpointLocation", "abfss://rawcat3@sagarcapstone3catalog.dfs.core.windows.net/stream_checkpoint_new_/")
      .outputMode("append")
      .trigger(processingTime="5 seconds")
      .table("sagar_cap3_cat1.streamin.stream_till_now")  
) 

In [0]:
%sql 
use catalog sagar_cap3_cat1;
create schema if not exists streamin;
drop table if exists stream_till_now 

In [0]:
%sql select count(*) from sagar_cap3_cat1.streamin.stream_till_now

count(*)
1733869
